# 03. Ticket Sentiment & Tone Analysis — via LLM Prompting (Azure OpenAI Responses API)

This notebook uses the same Azure AI Foundry + Responses API pattern as notebooks 01–02, but for a different
task: **sentiment and tone analysis**. The system prompt asks the model to identify each distinct concern
raised in a support ticket, score the sentiment/intensity of each one individually, and separately assess the
*overall* tone of the message.

This is free-form, per-aspect sentiment analysis driven entirely by prompt instructions — there is no fixed
output schema here (unlike notebook 02) and no confidence scores. Compare this conceptually to Azure AI
Language's **Sentiment Analysis** and **Opinion Mining** features, which return a much more rigid,
confidence-scored structure (see the exam tip below) — this repo doesn't currently have a notebook calling
that specific Language API, but it's worth knowing the contrast for AI-102.

**Difficulty:** Beginner


## Prerequisites

**Python packages** (already covered by the repo's root `requirements.txt`):
- `openai`, `azure-ai-projects`, `azure-identity`, `python-dotenv`

**Azure resources required:**
- An Azure AI Foundry project with a chat-capable model deployment

**Environment variables** (add to root `.env`):
```
AZURE_AI_PROJECT_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/api/projects/<your-project>
AZURE_AI_MODEL_DEPLOYMENT=<your-deployment-name>
```

Auth is via `DefaultAzureCredential` (`az login`).


## What You'll Learn

- How to prompt an LLM for multi-aspect (per-topic) sentiment rather than a single document-level label
- Why free-form sentiment output trades structure/consistency for nuance and explanation ("rationale")
- How this differs from Azure AI Language's Sentiment Analysis (document/sentence-level, fixed labels,
  confidence scores) and Opinion Mining (aspect-based sentiment with target/assessment pairs)
- When you'd reach for an LLM prompt vs. the dedicated Language service for sentiment


### Step 1 — Connect to the Azure AI Foundry project (same setup as notebooks 01–02)

💡 **Exam tip:** Azure AI Language's Sentiment Analysis API classifies each **sentence and the overall
document** into `positive`, `neutral`, `negative`, or `mixed`, each with a confidence score per label. Its
**Opinion Mining** feature (an option on the same API call, `show_opinion_mining=True`) goes further and
extracts *aspect-based* sentiment — pairs of a "target" (e.g. "VPN client") and an "assessment" (e.g.
"dropping", tagged negative) grounded in the exact text span. That's the closest dedicated-service equivalent
to what this notebook does with a free-form prompt.

🔄 **Alternatives:** None new at this step — see notebook 01 for client-setup alternatives.


In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
deployment_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT"]

project = AIProjectClient(
    endpoint=endpoint,
    credential=DefaultAzureCredential(),
)

openai = project.get_openai_client()

### Step 2 — Define the ticket text and the sentiment/tone analysis prompt

This ticket text is a slightly extended version of the one from notebooks 01–02 — it now includes a positive
remark about a specific support agent ("Marcus"), alongside the original frustrated complaint. This lets the
model demonstrate **mixed sentiment across distinct topics within one document**: frustration about the SLA
breach, but gratitude toward an individual.

The `system_prompt` asks for two levels of analysis:
1. Per-concern: sentiment + intensity score + rationale, for each distinct issue raised
2. Overall: one tone assessment for the message as a whole

💡 **Exam tip:** This "mixed sentiment in one document" scenario is exactly the case where Azure AI Language's
document-level sentiment would report `mixed`, while sentence-level sentiment would show the individual
sentences' true polarity — know that Language's response includes *both* granularities in a single API call,
not just one.

🔄 **Alternatives:** Ask the model to output structured JSON here too (as in notebook 02) if you need to feed
per-concern sentiment into downstream code rather than just printing it for a human to read.


In [ ]:
ticket_text = """
Subject: VPN disconnect issue - escalation needed

Hi team, this is Sarah Chen from Acme Logistics writing in again about
ticket TKT-1042. Our Gold-tier SLA promises a 4 hour response time, and
we are now at hour 6 with no update. The VPN client keeps dropping every
10 minutes on our Windows fleet since the rollout of CloudXeus Connect
v3.2 last Tuesday. If this isn't resolved by end of day Friday we will
be requesting the $500 SLA breach credit outlined in our contract.

That said, I do want to say thank you to Marcus on your team who called
me yesterday and was incredibly helpful in walking through workarounds.
"""

system_prompt = """
You are a sentiment and tone analysis engine for CloudXeus support tickets.
Identify each distinct concern or topic raised in the text, and for each one
report the sentiment, an intensity score, and a short rationale.
Separately, assess the overall tone of the message as a whole.
"""

### Step 3 — Call the Responses API and print the analysis

Same call shape as notebooks 01 and 03's earlier setup — no schema constraint here, so the model has full
freedom in how it formats the per-concern breakdown and the overall tone summary.

💡 **Exam tip:** Know the difference in what each approach *guarantees*: Azure AI Language's Sentiment
Analysis gives you a numeric confidence score per sentiment label (`positive_score`, `neutral_score`,
`negative_score`, summing to 1.0) for every sentence — deterministic, auditable, and consistent across runs.
This prompt-based approach gives you free-text intensity/rationale that reads better to humans but has no
such numeric guarantee and can vary between runs of the same input.

🔄 **Alternatives:** For production ticket triage where you need consistent, comparable sentiment scores across
thousands of tickets (e.g. to sort/filter/alert), the dedicated Language Sentiment Analysis + Opinion Mining
API is usually the better engineering choice; reach for LLM prompting when you need the nuanced, human-readable
"why" behind the sentiment.


In [ ]:
response = openai.responses.create(
    model=deployment_name,
    input=[
        {"type": "message", "role": "system", "content": system_prompt},
        {"type": "message", "role": "user", "content": ticket_text}
    ]
)

print(response.output_text)

## Summary

This notebook asked an LLM to break a support ticket into its distinct concerns, score the sentiment/intensity
of each, and separately assess overall tone — entirely through prompt instructions, with no fixed schema. The
result is a readable, nuanced analysis, in contrast to the fixed-format, confidence-scored output you'd get
from Azure AI Language's Sentiment Analysis and Opinion Mining features.


## Try It Yourself

1. **Easy:** Add a third, purely positive ticket text and rerun — compare how the model's "overall tone"
   assessment differs from the mixed-sentiment example.
2. **Intermediate:** Modify the system prompt to request a numeric intensity score on a fixed 1–5 scale for
   every concern, and a single overall sentiment label from a fixed enum (`positive`/`neutral`/`negative`/
   `mixed`) — moving this closer to how Azure AI Language structures its output.
3. **Advanced:** Combine this notebook's approach with notebook 02's structured-output technique: define a
   JSON Schema for `{"concerns": [{"topic": str, "sentiment": enum, "intensity": int, "rationale": str}],
   "overall_tone": enum}` and rerun with `strict: True` so the result is directly parseable.
